# Lab 4 - Basic Text Pre Processing

CISB5123 Text Analytics

In [17]:
import pandas as pd
import re
import emoji
import string
import nltk
from bs4 import BeautifulSoup
from autocorrect import Speller
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

# Initialize tools
spell = Speller(lang='en')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Dictionary of slang words
slang_dict = {
 "tbh": "to be honest",
 "omg": "oh my god",
 "lol": "laugh out loud",
 "idk": "I don't know",
 "brb": "be right back",
 "btw": "by the way",
 "imo": "in my opinion",
 "smh": "shaking my head",
 "fyi": "for your information",
 "np": "no problem",
 "ikr": "I know right",
 "asap": "as soon as possible",
 "bff": "best friend forever",
 "gg": "good game",
 "hmu": "hit me up",
 "rofl": "rolling on the floor laughing"
}

# Contractions dictionary
contractions_dict = {
 "wasn't": "was not",
 "isn't": "is not",
 "aren't": "are not",
 "weren't": "were not",
 "doesn't": "does not",
 "don't": "do not",
 "didn't": "did not",
 "can't": "cannot",
 "couldn't": "could not",
 "shouldn't": "should not",
 "wouldn't": "would not",
 "won't": "will not",
 "haven't": "have not",
 "hasn't": "has not",
 "hadn't": "had not",
 "i'm": "i am",
 "you're": "you are",
 "he's": "he is",
 "she's": "she is",
 "it's": "it is",
 "we're": "we are",
 "they're": "they are",
 "i've": "i have",
 "you've": "you have",
 "we've": "we have",
 "they've": "they have",
 "i'd": "i would",
 "you'd": "you would",
 "he'd": "he would",
 "she'd": "she would",
 "we'd": "we would",
 "they'd": "they would",
 "i'll": "i will",
 "you'll": "you will",
 "he'll": "he will",
 "she'll": "she will",
 "we'll": "we will",
 "they'll": "they will",
 "let's": "let us",
 "that's": "that is",
 "who's": "who is",
 "what's": "what is",
 "where's": "where is",
 "when's": "when is",
 "why's": "why is"
}

# Remove URLs
def remove_urls(text):
 return re.sub(r'http\S+|www\S+', '', text)

# Remove HTML tags
def remove_html(text):
 return BeautifulSoup(text, "html.parser").get_text()

# Remove emojis
def remove_emojis(text):
 return emoji.replace_emoji(text, replace='')

# Replace slang words
def replace_slang(text):

 escaped_slang_words = []

 for word in slang_dict.keys():
  escaped_word = re.escape(word)
  escaped_slang_words.append(escaped_word)

 slang_pattern = r'\b(' + '|'.join(escaped_slang_words) + r')\b'

 def replace_match(match):
  slang_word = match.group(0)
  return slang_dict[slang_word.lower()]

 replaced_text = re.sub(slang_pattern, replace_match, text, flags=re.IGNORECASE)

 return replaced_text

# Build contraction regex
escaped_contractions = []

for contraction in contractions_dict.keys():
 escaped_contraction = re.escape(contraction)
 escaped_contractions.append(escaped_contraction)

joined_contractions = "|".join(escaped_contractions)
contractions_pattern = r'\b(' + joined_contractions + r')\b'
compiled_pattern = re.compile(contractions_pattern, flags=re.IGNORECASE)

# Replace contractions
def replace_contractions(text):

 def replace_match(match):
  matched_word = match.group(0)
  lower_word = matched_word.lower()
  return contractions_dict[lower_word]

 expanded_text = compiled_pattern.sub(replace_match, text)

 return expanded_text

# Remove punctuation
def remove_punctuation(text):
 return text.translate(str.maketrans('', '', string.punctuation))

# Remove numbers
def remove_numbers(text):
 return re.sub(r'\d+', '', text)

# Spelling correction
def correct_spelling(text):
 return spell(text)

# Remove stopwords
def remove_stopwords(text):
 words = text.split()
 filtered_words = [word for word in words if word.lower() not in stop_words]
 return " ".join(filtered_words)

# POS tag mapping
def get_wordnet_pos(nltk_tag):
 if nltk_tag.startswith('J'):
  return wordnet.ADJ
 elif nltk_tag.startswith('V'):
  return wordnet.VERB
 elif nltk_tag.startswith('N'):
  return wordnet.NOUN
 elif nltk_tag.startswith('R'):
  return wordnet.ADV
 else:
  return wordnet.NOUN


# Lemmatization
def lemmatize_text(text):
 if not isinstance(text, str):
  return ""
 words = word_tokenize(text)
 pos_tags = pos_tag(words)
 lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
 return " ".join(lemmatized_words)

# Tokenization
def tokenize_text(text):
 if not isinstance(text, str):
  return []
 return word_tokenize(text)

# Full preprocessing pipeline
def preprocess_text(text):
 if not isinstance(text, str):
  return []
 text = text.lower()
 text = remove_urls(text)
 text = remove_html(text)
 text = remove_emojis(text)
 text = replace_slang(text)
 text = replace_contractions(text)
 text = remove_punctuation(text)
 text = remove_numbers(text)
 text = correct_spelling(text)
 text = remove_stopwords(text)
 text = lemmatize_text(text)
 text = tokenize_text(text)
 return text

# Load dataset
df = pd.read_csv("UNITENReview.csv")
# Remove missing reviews
df["Review"] = df["Review"].fillna("")
# Apply preprocessing
df["Processed"] = df["Review"].apply(preprocess_text)
# Save result
df.to_csv("Processed_Reviews2.csv", index=False)
# Show results
print(df[["Review", "Processed"]].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


                                              Review  \
0  Im happy with uniten actually, even the people...   
1  I’m having a pretty good time here, happy to m...   
2        a very neutral place in terms of everything   
3  I would say Uniten it's  a good university  bu...   
4   UNITEN is well-regarded, particularly for its...   

                                           Processed  
0      [im, happy, unite, actually, even, people, w]  
1  [i, ’, m, pretty, good, time, happy, meet, w, ...  
2                 [neutral, place, term, everything]  
3  [would, say, united, good, university, issue, ...  
4  [united, wellregarded, particularly, strong, e...  
